# 08 - Causal Circuit Surgery (Ablation & Intervention)

**SpectralCircuit-Former** research suite — T4 GPU (16GB) এ 100% চালানোর জন্য ডিজাইন করা। সেল উপর থেকে ক্রমানুসারে চালান। GPU: Runtime > Change runtime type > T4 GPU.

স্পেকট্রালভাবে চিহ্নিত head-গুলো কি সত্যিই **causally important**?
1. **Exhaustive ablation** — প্রতিটি head আলাদাভাবে zero-out করে accuracy-drop মাপা (ground truth causal importance)
2. **Spectral ranking** — শুধু spectral metrics দিয়ে head importance অনুমান করা যায় কিনা (Spearman correlation)
3. **Surgery** — top-3 causal heads zero করলে কী হয় vs bottom-3 vs random-3; boost (mask=2.0) করলে কী হয়

Runtime: T4-তে ~12-15 min।


In [ ]:
"""SpectralCircuit-Former — consolidated Colab edition (auto GPU/CPU)."""
import math
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OP, EQ = 0, 1
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.3})


def build_dataset(p=53, train_frac=0.4, seed=0, op="+"):
    g = torch.Generator().manual_seed(seed)
    pairs = torch.stack(torch.meshgrid(torch.arange(p), torch.arange(p), indexing="ij"), dim=-1).reshape(-1, 2)
    targets = (pairs[:, 0] + pairs[:, 1]) % p if op == "+" else (pairs[:, 0] * pairs[:, 1]) % p
    op_tok = OP if op == "+" else OP + 1
    seq = torch.stack([pairs[:, 0] + 3, torch.full((len(pairs),), op_tok),
                       pairs[:, 1] + 3, torch.full((len(pairs),), EQ)], dim=1)
    perm = torch.randperm(len(pairs), generator=g)
    n_train = int(len(pairs) * train_frac)
    return seq[perm[:n_train]], targets[perm[:n_train]], seq[perm[n_train:]], targets[perm[n_train:]]


def sym_laplacian(A):
    d = A.sum(dim=-2).clamp_min(1e-8).rsqrt()
    B = 0.5 * (A / d.unsqueeze(-2) + (A / d.unsqueeze(-1)).transpose(-1, -2))
    N = A.shape[-1]
    return torch.eye(N, device=A.device, dtype=A.dtype) - B, B


def fiedler_value(A, differentiable=False):
    with torch.set_grad_enabled(differentiable):
        L, _ = sym_laplacian(A)
        return torch.linalg.eigvalsh(L)[..., 1]


def spectral_entropy(A):
    _, B = sym_laplacian(A)
    ev = torch.linalg.eigvalsh(B).abs().clamp_min(1e-12)
    p = ev / ev.sum(dim=-1, keepdim=True)
    N = A.shape[-1]
    return -(p * p.log()).sum(dim=-1) / math.log(N)


def mean_row_entropy(A):
    q = A.clamp_min(1e-12)
    return -(q * q.log()).sum(dim=-1).mean(dim=-1) / math.log(A.shape[-1])


def sink_fraction(A):
    return A.max(dim=-2).values.mean(dim=-1)


def effective_rank(X):
    Xc = X - X.mean(dim=-2, keepdim=True)
    s = torch.linalg.svdvals(Xc)
    return (s.sum(dim=-1) ** 2) / s.pow(2).sum(dim=-1).clamp_min(1e-12)


def fiedler_regularizer(A_probe, tau):
    L, _ = sym_laplacian(A_probe)
    lam2 = torch.linalg.eigvalsh(L)[..., 1]
    return torch.relu(tau - lam2).mean(), lam2.detach()


class Block(nn.Module):
    def __init__(self, d_model, n_heads, mlp_mult=4, use_residual=True):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.use_residual = use_residual
        self.ln1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, mlp_mult * d_model)
        self.fc2 = nn.Linear(mlp_mult * d_model, d_model)

    def forward(self, x, head_mask=None):
        B, N, D = x.shape
        h = self.ln1(x)
        qkv = self.qkv(h).view(B, N, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        att = (q @ k.transpose(-1, -2)) / math.sqrt(self.d_head)
        mask = torch.triu(torch.ones(N, N, device=x.device, dtype=torch.bool), diagonal=1)
        A = att.masked_fill(mask, float("-inf")).softmax(dim=-1)
        y = (A @ v).transpose(1, 2).reshape(B, N, D)
        if head_mask is not None:
            m = head_mask.repeat_interleave(self.d_head).view(1, 1, D)
            y = y * m
        x = self.proj(y) + (x if self.use_residual else 0)
        x = self.fc2(F.gelu(self.fc1(self.ln2(x)))) + (x if self.use_residual else 0)
        return x, A


class TinyTransformer(nn.Module):
    def __init__(self, vocab, d_model=128, n_layers=2, n_heads=4, max_len=16, use_residual=True):
        super().__init__()
        self.n_layers = n_layers
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList(Block(d_model, n_heads, use_residual=use_residual) for _ in range(n_layers))
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)

    def forward(self, idx, head_mask=None, full_logits=False):
        B, N = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(N, device=idx.device))
        attn_maps = []
        for i, blk in enumerate(self.blocks):
            hm = None if head_mask is None else head_mask[i]
            x, A = blk(x, head_mask=hm)
            attn_maps.append(A)
        z = self.head(self.ln_f(x))
        if full_logits:
            return z, attn_maps, x
        return z[:, -1], attn_maps, x

    def param_groups(self, wd):
        decay, no_decay = [], []
        for name, p in self.named_parameters():
            (no_decay if (p.ndim <= 1 or "ln" in name) else decay).append(p)
        return [{"params": decay, "weight_decay": wd}, {"params": no_decay, "weight_decay": 0.0}]


@torch.no_grad()
def evaluate(model, seq, targets, bs=4096):
    correct = 0
    for i in range(0, len(seq), bs):
        logits, _, _ = model(seq[i:i + bs])
        correct += (logits.argmax(-1) == targets[i:i + bs]).sum().item()
    return correct / len(seq)


@torch.no_grad()
def spectral_probe(model, probe):
    _, attn_maps, hidden = model(probe)
    L, H, N = len(attn_maps), attn_maps[0].shape[1], attn_maps[0].shape[-1]
    lam2 = np.zeros((L, H)); sent = np.zeros((L, H)); rent = np.zeros((L, H))
    sink = np.zeros((L, H)); spectra = np.zeros((L, H, N))
    for l, A in enumerate(attn_maps):
        lam2[l] = fiedler_value(A).mean(dim=0).cpu().numpy()
        sent[l] = spectral_entropy(A).mean(dim=0).cpu().numpy()
        rent[l] = mean_row_entropy(A).mean(dim=0).cpu().numpy()
        sink[l] = sink_fraction(A).mean(dim=0).cpu().numpy()
        _, B = sym_laplacian(A)
        spectra[l] = torch.linalg.eigvalsh(B).mean(dim=0).cpu().numpy()
    h = model.tok(probe) + model.pos(torch.arange(probe.shape[1], device=probe.device))
    eff_ranks = [float(effective_rank(h).mean())]
    for blk in model.blocks:
        h, _ = blk(h)
        eff_ranks.append(float(effective_rank(h).mean()))
    return lam2, sent, rent, sink, spectra, eff_ranks


def train_run(seed, regularize=False, p=53, train_frac=0.4, d_model=128, n_layers=2,
              n_heads=4, lr=1e-3, wd=2.0, max_steps=12000, log_every=100,
              tau=0.35, beta1=1.0, dataset=None, op="+", use_residual=True):
    torch.manual_seed(seed)
    if dataset is None:
        dataset = build_dataset(p, train_frac, seed, op)
    train_x, train_y, test_x, test_y = (t.to(DEVICE) for t in dataset)
    model = TinyTransformer(p + 3, d_model, n_layers, n_heads, max_len=train_x.shape[1],
                            use_residual=use_residual).to(DEVICE)
    opt = torch.optim.AdamW(model.param_groups(wd), lr=lr, betas=(0.9, 0.98))
    g = torch.Generator().manual_seed(seed + 999)
    idx = torch.randperm(len(train_x), generator=g)[:64].to(DEVICE)
    probe = train_x[idx]
    log = {"step": [], "train_loss": [], "train_acc": [], "test_acc": [], "wall_s": [],
           "lam2": [], "spec_ent": [], "row_ent": [], "sink": [], "spectra": [], "eff_rank": [],
           "A_mean": []}
    transition_step = None
    t0 = time.time()
    for step in range(1, max_steps + 1):
        logits, attn_maps, hidden = model(train_x)
        loss = F.cross_entropy(logits, train_y)
        if regularize:
            reg, _ = fiedler_regularizer(torch.stack(attn_maps, dim=1)[probe], tau)
            loss = loss + beta1 * reg
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % log_every == 0 or step == 1:
            with torch.no_grad():
                tr_logits, _, _ = model(train_x)
                tr_acc = (tr_logits.argmax(-1) == train_y).float().mean().item()
            te_acc = evaluate(model, test_x, test_y)
            lam2, sent, rent, sink, spectra, eff = spectral_probe(model, probe)
            log["step"].append(step); log["train_loss"].append(float(loss.detach()))
            log["train_acc"].append(tr_acc); log["test_acc"].append(te_acc)
            log["wall_s"].append(round(time.time() - t0, 1))
            log["lam2"].append(lam2.tolist()); log["spec_ent"].append(sent.tolist())
            log["row_ent"].append(rent.tolist()); log["sink"].append(sink.tolist())
            log["spectra"].append(spectra.tolist()); log["eff_rank"].append(eff)
            log["A_mean"].append(torch.stack(attn_maps, dim=1).mean(dim=0).cpu().tolist())
            print(f"  step {step:5d} | loss {loss.item():.4f} | test_acc {te_acc:.3f} | lam2 {lam2.mean():.3f} | {time.time()-t0:.0f}s", flush=True)
            if te_acc >= 0.95 and transition_step is None:
                transition_step = step
            if te_acc >= 0.999 and step >= (transition_step or 0) + 2000:
                break
    return {"config": {"seed": seed, "regularize": regularize, "p": p, "tau": tau, "op": op,
                       "use_residual": use_residual, "n_layers": n_layers},
            "transition_step": transition_step, "final_test_acc": log["test_acc"][-1],
            "steps_run": log["step"][-1], "log": log, "model": model}
print('core ready on', DEVICE)



In [ ]:
"""Notebook-specific extra functions (curvature, clustering, tasks, circuits, ablation)."""
import numpy as np
import torch
import torch.nn.functional as F


def marchenko_pastur_pdf(x, q):
    x = np.asarray(x, float)
    lam_m, lam_p = (1 - np.sqrt(q)) ** 2, (1 + np.sqrt(q)) ** 2
    y = np.zeros_like(x)
    m = (x >= lam_m) & (x <= lam_p)
    y[m] = np.sqrt(np.maximum((lam_p - x[m]) * (x[m] - lam_m), 0)) / (2 * np.pi * q * x[m])
    return y


def fiedler_partition(A):
    """2-way spectral clustering of a (N,N) attention matrix via Fiedler vector sign."""
    d = A.sum(-2).clamp_min(1e-8).rsqrt()
    B = 0.5 * (A / d.unsqueeze(-2) + (A / d.unsqueeze(-1)).transpose(-1, -2))
    L = torch.eye(A.shape[-1], device=A.device) - B
    evals, evecs = torch.linalg.eigh(L)
    partition = (evecs[:, 1] > 0).long()
    return partition, evals


def _hop_distance(A):
    An = (np.asarray(A, float) > 0.01).astype(float)
    An = np.maximum(An, An.T)
    An = np.maximum(An, np.eye(len(An)))
    D = np.full((len(An), len(An)), np.inf)
    for s in range(len(An)):
        D[s, s] = 0.0
        seen = np.zeros(len(An), bool); seen[s] = True
        frontier = np.array([s])
        d = 0
        while len(frontier):
            d += 1
            nxt = np.unique(np.where(An[frontier].any(0))[0])
            nxt = nxt[~seen[nxt]]
            if not len(nxt):
                break
            D[s, nxt] = d
            seen[nxt] = True
            frontier = nxt
    D[~np.isfinite(D)] = len(An)
    return D


def ollivier_ricci(A, alpha=0.5):
    """Exact Ollivier-Ricci curvature per edge using lazy random walks and W1
    over the hop-distance ground metric (small graphs only, N <= ~16)."""
    from scipy.optimize import linprog
    A = np.asarray(A, float)
    N = len(A)
    P = A / np.maximum(A.sum(-1, keepdims=True), 1e-12)
    Dg = _hop_distance(A)
    kappa = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i == j or P[i, j] < 1e-6:
                continue
            mi = np.zeros(N); mi[i] = 1 - alpha; mi += alpha * P[i]
            mj = np.zeros(N); mj[j] = 1 - alpha; mj += alpha * P[j]
            c = Dg.reshape(-1)
            Amat = np.kron(np.eye(N), np.ones((1, N))) - np.kron(np.ones((1, N)), np.eye(N))
            res = linprog(c, A_eq=Amat, b_eq=mi - mj, bounds=[(0, None)] * (N * N), method="highs")
            if res.success:
                kappa[i, j] = 1 - res.fun / max(Dg[i, j], 1e-9)
    return kappa


def forman_ricci(A):
    """Simplified weighted Forman-Ricci on the symmetrized attention graph."""
    W = 0.5 * (np.asarray(A, float) + np.asarray(A, float).T)
    np.fill_diagonal(W, 0)
    N = len(W)
    F = np.zeros((N, N))
    for i in range(N):
        for j in range(i + 1, N):
            if W[i, j] < 1e-9:
                continue
            w_e = W[i, j]
            s_i = W[i].sum() + 1e-12; s_j = W[j].sum() + 1e-12
            t_i = sum(np.sqrt(w_e * W[i, k]) for k in range(N) if k != j and W[i, k] > 1e-9)
            t_j = sum(np.sqrt(w_e * W[j, k]) for k in range(N) if k != i and W[j, k] > 1e-9)
            F[i, j] = F[j, i] = w_e * (w_e / s_i + w_e / s_j - t_i / s_i - t_j / s_j)
    return F


def head_roles(model, probe, feature_names=None):
    """Classify heads by dominant attention behaviour. Returns dict of arrays."""
    with torch.no_grad():
        _, attn_maps, _ = model(probe)
    rows = []
    for l, A in enumerate(attn_maps):
        Am = A.mean(dim=0)
        for h in range(Am.shape[0]):
            col = Am[h].mean(0)
            dom = int(col.argmax())
            rows.append({"layer": l, "head": h, "dominant_col": dom,
                         "lam2": float(fiedler_value(Am[h:h + 1])[0]),
                         "spec_ent": float(spectral_entropy(Am[h:h + 1])[0]),
                         "sink": float(sink_fraction(Am[h:h + 1])[0])})
    return rows


def layer_circuit_corr(model, probe):
    """Head-to-head attention-pattern cosine similarity between consecutive layers
    (candidates for the multi-layer circuit DAG)."""
    with torch.no_grad():
        _, attn_maps, _ = model(probe)
    L = len(attn_maps)
    H = attn_maps[0].shape[1]
    C = np.zeros((L - 1, H, H))
    for l in range(L - 1):
        X = attn_maps[l].mean(0).reshape(H, -1)
        Y = attn_maps[l + 1].mean(0).reshape(H, -1)
        X = X - X.mean(1, keepdims=True); Y = Y - Y.mean(1, keepdims=True)
        C[l] = (X @ Y.T) / (np.linalg.norm(X, axis=1, keepdims=True) *
                            np.linalg.norm(Y, axis=1, keepdims=True).T + 1e-9)
    return C


@torch.no_grad()
def ablation_importance(model, seq, targets, bs=4096):
    """Per-head causal importance: accuracy drop when head output is zeroed."""
    L = model.n_layers
    H = model.blocks[0].n_heads
    base = evaluate(model, seq, targets, bs)
    imp = np.zeros((L, H))
    for l in range(L):
        for h in range(H):
            hm = torch.ones(L, H, device=seq.device)
            hm[l, h] = 0.0
            correct, n = 0, 0
            for i in range(0, len(seq), bs):
                logits, _, _ = model(seq[i:i + bs], head_mask=hm)
                correct += (logits.argmax(-1) == targets[i:i + bs]).sum().item()
                n += len(seq[i:i + bs])
            imp[l, h] = base - correct / n
    return base, imp


@torch.no_grad()
def head_mask_eval(model, seq, targets, head_mask, bs=4096):
    correct, n = 0, 0
    for i in range(0, len(seq), bs):
        logits, _, _ = model(seq[i:i + bs], head_mask=head_mask)
        correct += (logits.argmax(-1) == targets[i:i + bs]).sum().item()
        n += len(seq[i:i + bs])
    return correct / n


def rank_profile(model, probe):
    """Per-layer effective rank and mean pairwise token cosine similarity."""
    with torch.no_grad():
        N = probe.shape[1]
        h = model.tok(probe) + model.pos(torch.arange(N, device=probe.device))
        eff = [float(effective_rank(h).mean())]
        cos = []
        hc = h - h.mean(dim=1, keepdim=True)
        hc = hc / hc.norm(dim=-1, keepdim=True).clamp_min(1e-9)
        cos.append(float((hc @ hc.transpose(-1, -2)).mean()))
        for blk in model.blocks:
            h, _ = blk(h)
            eff.append(float(effective_rank(h).mean()))
            hc = h - h.mean(dim=1, keepdim=True)
            hc = hc / hc.norm(dim=-1, keepdim=True).clamp_min(1e-9)
            cos.append(float((hc @ hc.transpose(-1, -2)).mean()))
    return np.array(eff), np.array(cos)


def dyck1_dataset(n_samples, max_prefix=8, seed=0, train_frac=0.7):
    """Dyck-1 balanced parentheses: sequence + next-token targets (LM style).
    Vocab: 0='(' 1=')' 2=BOS. Target at position t is token t+1 (shifted)."""
    g = np.random.default_rng(seed)
    seqs, tgts = [], []
    for _ in range(n_samples):
        depth, s = 0, [2]
        for _ in range(max_prefix):
            if depth == 0:
                s.append(0); depth += 1
            elif depth < max_prefix // 2 and g.random() < 0.55:
                s.append(0); depth += 1
            else:
                s.append(1); depth -= 1
        s.append(1); depth -= 1
        seqs.append(s[:-1]); tgts.append(s[1:])
    seqs = torch.tensor(seqs); tgts = torch.tensor(tgts)
    perm = torch.randperm(n_samples, generator=torch.Generator().manual_seed(seed))
    n_train = int(n_samples * train_frac)
    return seqs[perm[:n_train]], tgts[perm[:n_train]], seqs[perm[n_train:]], tgts[perm[n_train:]]


def reach_dataset(n_graphs=40, n_nodes=8, p_edge=0.4, n_queries=150, k=2, seed=0, train_frac=0.7):
    """k-hop reachability: fixed random graph per dataset; query [u, ?, v] -> bit.
    Vocab: 0..n_nodes-1 = nodes, n_nodes = QUERY token, n_nodes+1 = unused; target in {0,1}."""
    g = np.random.default_rng(seed)
    G = (g.random((n_nodes, n_nodes)) < p_edge).astype(float)
    np.fill_diagonal(G, 1.0)
    reach = G.copy()
    for _ in range(k - 1):
        reach = ((reach @ G) > 0).astype(float)
    pairs = np.array([(u, v) for u in range(n_nodes) for v in range(n_nodes)])
    sel = g.choice(len(pairs), size=min(n_queries, len(pairs)), replace=False)
    seq = torch.stack([torch.tensor(pairs[sel, 0]), torch.full((len(sel),), n_nodes),
                       torch.tensor(pairs[sel, 1])], dim=1)
    tgt = torch.tensor([int(reach[u, v]) for u, v in pairs[sel]])
    perm = torch.randperm(len(sel), generator=torch.Generator().manual_seed(seed))
    n_train = int(len(sel) * train_frac)
    return seq[perm[:n_train]], tgt[perm[:n_train]], seq[perm[n_train:]], tgt[perm[n_train:]]


def train_lm_dyck(train_x, train_y, test_x, test_y, d_model=64, n_layers=2, n_heads=4,
                  lr=1e-3, wd=1.0, max_steps=4000, log_every=200):
    """Next-token LM training for Dyck-1 with full-sequence cross-entropy.
    Requires TinyTransformer, DEVICE, spectral_probe in the namespace (notebook core cell)."""
    import torch.nn.functional as Fn
    torch.manual_seed(0)
    model = TinyTransformer(3, d_model, n_layers, n_heads, max_len=train_x.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.param_groups(wd), lr=lr, betas=(0.9, 0.98))
    probe = train_x[:64]
    log = {"step": [], "loss": [], "val_acc": [], "lam2": [], "spec_ent": []}
    for step in range(1, max_steps + 1):
        logits, attn_maps, _ = model(train_x, full_logits=True)
        loss = Fn.cross_entropy(logits.reshape(-1, 3), train_y.reshape(-1))
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        if step % log_every == 0 or step == 1:
            with torch.no_grad():
                tl, am, _ = model(test_x, full_logits=True)
                acc = (tl.argmax(-1) == test_y).float().mean().item()
            lam2, sent, _, _, _, _ = spectral_probe(model, probe)
            log["step"].append(step); log["loss"].append(float(loss)); log["val_acc"].append(acc)
            log["lam2"].append(float(lam2.mean())); log["spec_ent"].append(float(sent.mean()))
            print(f"  step {step:5d} | lm_loss {loss.item():.4f} | val_next_token_acc {acc:.3f} | lam2 {lam2.mean():.3f}", flush=True)
    return model, log
print('extras ready')



# 08 — Causal Circuit Surgery (Ablation & Intervention)
স্পেকট্রালভাবে চিহ্নিত head-গুলো কি সত্যিই **causally important**?
1. **Exhaustive ablation** — প্রতিটি head আলাদাভাবে zero-out করে accuracy-drop মাপা (ground truth causal importance)
2. **Spectral ranking** — শুধু spectral metrics দিয়ে head importance অনুমান করা যায় কিনা (Spearman correlation)
3. **Surgery** — top-3 causal heads zero করলে কী হয় vs bottom-3 vs random-3; boost (mask=2.0) করলে কী হয়

Runtime: T4-তে ~12-15 min।


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats

res = train_run(seed=0, p=53, d_model=128, n_layers=2, n_heads=4, max_steps=12000)
model = res["model"]
_, _, tx, ty = build_dataset(p=53, train_frac=0.4, seed=0)
tx, ty = tx.to(DEVICE), ty.to(DEVICE)
base = res["final_test_acc"]
print("base test acc:", round(base, 4))


In [ ]:
base_eval, imp = ablation_importance(model, tx, ty)
L, H = imp.shape
fig, ax = plt.subplots(figsize=(5, 3))
im = ax.imshow(imp, cmap="Reds")
ax.set_xlabel("head"); ax.set_ylabel("layer")
ax.set_title("FIG 8.1 — Causal importance (accuracy drop when head zeroed)")
for l in range(L):
    for h in range(H):
        ax.text(h, l, f"{imp[l, h]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="acc drop"); plt.show()

flat = [(l, h, imp[l, h]) for l in range(L) for h in range(H)]
print("\nHead ranking by causal importance (top 5):")
for l, h, v in sorted(flat, key=lambda x: -x[2])[:5]:
    print(f"  L{l}H{h}: drop = {v:.4f}")

roles = head_roles(model, tx[:64])
lam2 = np.array([r["lam2"] for r in roles]).reshape(L, H)
sink = np.array([r["sink"] for r in roles]).reshape(L, H)
rho_l2 = stats.spearmanr(imp.flatten(), lam2.flatten()).statistic
rho_sink = stats.spearmanr(imp.flatten(), sink.flatten()).statistic
print(f"\nSpearman(causal importance, head lambda2)  = {rho_l2:+.3f}")
print(f"Spearman(causal importance, sink fraction) = {rho_sink:+.3f}")
print("-> Spectral metrics শুধু দেখে দেখে causal head চেনা যায় কিনা তার পরিমাপ।")


In [ ]:
top3 = [(l, h) for l, h, v in sorted(flat, key=lambda x: -x[2])[:3]]
bot3 = [(l, h) for l, h, v in sorted(flat, key=lambda x: x[2])[:3]]
rng = np.random.default_rng(0)
all_heads = [(l, h) for l in range(L) for h in range(H)]
rand3 = [rng.permutation(all_heads)[:3].tolist() for _ in range(5)]

def surgery(pairs, value):
    hm = torch.ones(L, H, device=DEVICE)
    for (l, h) in pairs:
        hm[l, h] = value
    return head_mask_eval(model, tx, ty, hm)

acc_top0 = surgery(top3, 0.0)
acc_bot0 = surgery(bot3, 0.0)
acc_rand0 = float(np.mean([surgery(p, 0.0) for p in rand3]))
acc_top2 = surgery(top3, 2.0)

df = pd.DataFrame([
    {"intervention": "baseline (no surgery)", "acc": base},
    {"intervention": "zero top-3 causal heads", "acc": acc_top0},
    {"intervention": "zero bottom-3 heads", "acc": acc_bot0},
    {"intervention": "zero random-3 heads (avg of 5)", "acc": acc_rand0},
    {"intervention": "boost top-3 (mask=2.0)", "acc": acc_top2},
])
print(df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 3.2))
colors = ["gray", "crimson", "seagreen", "gold", "royalblue"]
ax.bar(df["intervention"], df["acc"], color=colors)
ax.set_ylim(0, 1.05); ax.set_ylabel("test accuracy")
ax.set_title("FIG 8.2 — Causal circuit surgery")
ax.tick_params(axis="x", rotation=12)
for i, v in enumerate(df["acc"]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=8)
plt.show()
print("\nExpected: top-3 causal heads zero করলে accuracy ধসে পড়ে, bottom/random এ প্রায় অপরিবর্তিত")
print("— এটাই 'discovered circuit is causally necessary' দাবির প্রমাণ। Boost-এ error pattern বদলায়।")


---
*Run order: উপরের সব code cell উপর থেকে নিচে চালান। প্রশ্ন হলে FINDINGS.md দেখুন (repo root)।*
